In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

# =========================
# 0) 参数集中区（你只改这里）
# =========================
CONFIG = {
    "raw_file_path": "BAT_Heat_Log_Data_2026_05_21_17_49_41.csv",
    "fs": 50,  # 采样率 Hz

    # —— 滤波器设计参数——
    # filter_type: "lowpass" | "highpass" | "bandpass" | "bandstop"
    "filter": {
        "design": "butter",       # "butter"（推荐先用）/ 你以后也可以扩展 cheby、ellip
        "filter_type": "bandpass",
        "order": 1,               # Butterworth 阶数（注意：bandpass 下阶数解释见下）
        "lowcut_hz": 0.1,         # 带通下截止（Hz）
        "highcut_hz": 1,       # 带通上截止（Hz）  
        "steady_init_iter": 5000  # 复现 C++ initFilterSteadyState 的迭代次数
    },

    # 绘图参数（可选）
    "plot": {
        "alpha_raw": 0.35,
        "lw_filt": 2.0
    }
}


# =========================
# 1) 统一颜色规则（Accel & Gyro：x蓝 y绿 z红）
# =========================
axis_colors = {'x': 'blue', 'y': 'green', 'z': 'red'}

ACC_COLORS = {'Accel_x': axis_colors['x'], 'Accel_y': axis_colors['y'], 'Accel_z': axis_colors['z']}
GYRO_COLORS = {'gyro_x': axis_colors['x'],  'gyro_y': axis_colors['y'],  'gyro_z': axis_colors['z']}

RAW_IMU_COLS = ['Accel_x','Accel_y','Accel_z','gyro_x','gyro_y','gyro_z']

print("✅ 配置完成")
print("文件:", CONFIG["raw_file_path"])
print("fs:", CONFIG["fs"], "Hz")
print("滤波:", CONFIG["filter"])


from scipy import signal

def design_sos_from_config(cfg):
    fs = cfg["fs"]
    fcfg = cfg["filter"]
    ftype = fcfg["filter_type"].lower()
    order = int(fcfg["order"])

    nyq = fs / 2.0

    # 频率合法性检查（别让你踩坑）
    if ftype in ("bandpass", "bandstop"):
        low = float(fcfg["lowcut_hz"])
        high = float(fcfg["highcut_hz"])
        if not (0 < low < high < nyq):
            raise ValueError(f"band* 截止频率必须满足 0 < lowcut < highcut < fs/2({nyq}). 当前: {low}, {high}")
        wn = [low, high]
    else:
        # lowpass / highpass
        cut = float(fcfg.get("cutoff_hz", fcfg.get("highcut_hz", 5.0)))
        if not (0 < cut < nyq):
            raise ValueError(f"{ftype} 截止频率必须满足 0 < cutoff < fs/2({nyq}). 当前: {cut}")
        wn = cut

    # Butterworth SOS
    if fcfg["design"].lower() == "butter":
        sos = signal.butter(order, wn, btype=ftype, fs=fs, output="sos")
    
    else:
        raise NotImplementedError("目前只实现 butter，你要 cheby/ellip 我再给你加。")

    return sos

SOS_AUTO = design_sos_from_config(CONFIG)
print("✅ SOS 生成完成，stage 数:", SOS_AUTO.shape[0])



df = pd.read_csv(
    CONFIG["raw_file_path"],
    sep="\s+|,",        # 空格/Tab/逗号 全能识别
    engine="python",    # 必须保留
    skipinitialspace=True  # 忽略多余空格
)

df.columns = df.columns.str.strip()
df = df.loc[:, ~df.columns.str.contains(r'^Unnamed', regex=True)]
df = df.replace(r'^\s*$', np.nan, regex=True)

df_imu = df[RAW_IMU_COLS].copy()
for c in RAW_IMU_COLS:
    df_imu[c] = pd.to_numeric(df_imu[c], errors='coerce')

df_imu = df_imu.dropna(subset=RAW_IMU_COLS).reset_index(drop=True)
df_imu["time_sec"] = np.arange(len(df_imu), dtype=np.int64) / CONFIG["fs"]

print("数据长度:", len(df_imu))
print("总时长(s):", df_imu["time_sec"].iloc[-1])


class SosState:
    def __init__(self):
        self.z1 = 0.0
        self.z2 = 0.0

class FilterChannel:
    def __init__(self, sos_coeffs):
        self.sos_coeffs = np.array(sos_coeffs, dtype=np.float64)
        self.num_stages = self.sos_coeffs.shape[0]
        self.states = [SosState() for _ in range(self.num_stages)]

    def reset(self):
        for s in self.states:
            s.z1 = 0.0
            s.z2 = 0.0

def sos_filter_single(sec, x, state):
    b0, b1, b2, a0, a1, a2 = sec
    out = b0 * x + state.z1
    state.z1 = b1 * x - a1 * out + state.z2
    state.z2 = b2 * x - a2 * out
    return out

def sos_filter_cascade(channel, x):
    y = x
    for k in range(channel.num_stages):
        y = sos_filter_single(channel.sos_coeffs[k], y, channel.states[k])
    return y

def init_filter_steady_state(channel, dc_offset, n_iter=5000):
    channel.reset()
    for _ in range(int(n_iter)):
        sos_filter_cascade(channel, dc_offset)

def apply_sos_to_h(df_in, sos_coeffs, steady_iter):
    out = df_in.copy()
    channels = {col: FilterChannel(sos_coeffs) for col in RAW_IMU_COLS}

    # 第一帧稳态初始化
    for col, ch in channels.items():
        init_filter_steady_state(ch, out[col].iloc[0], n_iter=steady_iter)

    # 逐点滤波，写回 *_h
    for col, ch in channels.items():
        out[col + "_h"] = [sos_filter_cascade(ch, x) for x in out[col].to_numpy(np.float64)]

    return out

df_proc = apply_sos_to_h(
    df_imu,
    sos_coeffs=SOS_AUTO,
    steady_iter=CONFIG["filter"]["steady_init_iter"]
)

print("✅ 已生成:", [c + "_h" for c in RAW_IMU_COLS])





plt.figure(figsize=(14, 7))

# 选择滤波后列：优先 *_h_py，没有就用 *_h
acc_h_cols = ['Accel_x_h_py', 'Accel_y_h_py', 'Accel_z_h_py']
if not all(c in df_proc.columns for c in acc_h_cols):
    acc_h_cols = ['Accel_x_h', 'Accel_y_h', 'Accel_z_h']

# ---------- 原始（raw） ----------
plt.plot(df_proc['time_sec'], df_proc['Accel_x'],
         label='Accel_x (raw)', color=axis_colors['x'], linewidth=0.8, alpha=0.35)
plt.plot(df_proc['time_sec'], df_proc['Accel_y'],
         label='Accel_y (raw)', color=axis_colors['y'], linewidth=0.8, alpha=0.35)
plt.plot(df_proc['time_sec'], df_proc['Accel_z'],
         label='Accel_z (raw)', color=axis_colors['z'], linewidth=0.8, alpha=0.35)

# ---------- 滤波后（filtered） ----------
plt.plot(df_proc['time_sec'], df_proc[acc_h_cols[0]],
         label=f'{acc_h_cols[0]} (filtered)', color=axis_colors['x'], linewidth=1.8)
plt.plot(df_proc['time_sec'], df_proc[acc_h_cols[1]],
         label=f'{acc_h_cols[1]} (filtered)', color=axis_colors['y'], linewidth=1.8)
plt.plot(df_proc['time_sec'], df_proc[acc_h_cols[2]],
         label=f'{acc_h_cols[2]} (filtered)', color=axis_colors['z'], linewidth=1.8)

plt.xlabel("Time (s)")
plt.ylabel("Accel")
plt.title("Accelerometer: Raw vs Filtered")
plt.xlim(0, 120)
plt.ylim(-1500, 1500)
plt.legend(ncol=2)
plt.grid(True)
plt.show()



plt.figure(figsize=(14, 7))

# 选择滤波后列：优先 *_h_py，没有就用 *_h
gyro_h_cols = ['gyro_x_h_py', 'gyro_y_h_py', 'gyro_z_h_py']
if not all(c in df_proc.columns for c in gyro_h_cols):
    gyro_h_cols = ['gyro_x_h', 'gyro_y_h', 'gyro_z_h']

# ---------- 原始（raw） ----------
plt.plot(
    df_proc['time_sec'],
    df_proc['gyro_x'],
    label='gyro_x (raw)',
    color=axis_colors['x'],
    linewidth=0.8,
    alpha=0.35
)
plt.plot(
    df_proc['time_sec'],
    df_proc['gyro_y'],
    label='gyro_y (raw)',
    color=axis_colors['y'],
    linewidth=0.8,
    alpha=0.35
)
plt.plot(
    df_proc['time_sec'],
    df_proc['gyro_z'],
    label='gyro_z (raw)',
    color=axis_colors['z'],
    linewidth=0.8,
    alpha=0.35
)

# ---------- 滤波后（filtered） ----------
plt.plot(
    df_proc['time_sec'],
    df_proc[gyro_h_cols[0]],
    label=f'{gyro_h_cols[0]} (filtered)',
    color=axis_colors['x'],
    linewidth=1.8
)
plt.plot(
    df_proc['time_sec'],
    df_proc[gyro_h_cols[1]],
    label=f'{gyro_h_cols[1]} (filtered)',
    color=axis_colors['y'],
    linewidth=1.8
)
plt.plot(
    df_proc['time_sec'],
    df_proc[gyro_h_cols[2]],
    label=f'{gyro_h_cols[2]} (filtered)',
    color=axis_colors['z'],
    linewidth=1.8
)

plt.xlabel("Time (s)")
plt.ylabel("Gyro")
plt.title("Gyroscope: Raw vs Filtered")
plt.xlim(0, 120)

# ✅ 这里的 y 轴范围你可以按你数据量级改
plt.ylim(-250, 250)

plt.legend(ncol=2)
plt.grid(True)
plt.show()


<>:88: SyntaxWarning: invalid escape sequence '\s'
<>:88: SyntaxWarning: invalid escape sequence '\s'
C:\Users\85245579\AppData\Local\Temp\ipykernel_21524\1856125858.py:88: SyntaxWarning: invalid escape sequence '\s'
  sep="\s+|,",        # 空格/Tab/逗号 全能识别


✅ 配置完成
文件: BAT_Heat_Log_Data_2026_05_27_21_01_55.csv
fs: 50 Hz
滤波: {'design': 'butter', 'filter_type': 'bandpass', 'order': 1, 'lowcut_hz': 0.1, 'highcut_hz': 1, 'steady_init_iter': 5000}
✅ SOS 生成完成，stage 数: 1


FileNotFoundError: [Errno 2] No such file or directory: 'BAT_Heat_Log_Data_2026_05_27_21_01_55.csv'

In [ ]:
# ============================================================
# 补充实验：
# 1) 双向滤波（sosfiltfilt）——观察是否消除相位差（零相位）
# 2) 不做滤波器初始化（steady_init_iter=0）——观察开头是否产生震荡/过渡过程
#
# 说明：你当前 notebook 里复现的 C++ SOS 逐点滤波（sos_filter_cascade + 状态 z1/z2）
# 是【正向/因果滤波】（forward, single-pass IIR），不是双向滤波。
# ============================================================
import matplotlib as mpl
from scipy import signal

# 解决图例中文显示为方框：优先使用 Windows 常见中文字体
mpl.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Arial Unicode MS', 'Noto Sans CJK SC']
mpl.rcParams['axes.unicode_minus'] = False

fs = CONFIG["fs"]
sos = SOS_AUTO
alpha_raw = CONFIG.get("plot", {}).get("alpha_raw", 0.35)
lw_filt = CONFIG.get("plot", {}).get("lw_filt", 2.0)

print("✅ 滤波方向判断：C++/本Notebook 的逐点 SOS IIR = 正向(因果)滤波；双向滤波需要前向+反向两次处理（sosfiltfilt）。")

# ------------------------------
# 1) 双向滤波（零相位）：sosfiltfilt
# ------------------------------
df_compare = df_proc.copy()
for col in RAW_IMU_COLS:
    x = df_compare[col].to_numpy(np.float64)
    df_compare[col + "_bi"] = signal.sosfiltfilt(sos, x)

# ------------------------------
# 2) 不做初始化：steady_init_iter=0
# ------------------------------
df_noinit = apply_sos_to_h(df_imu, sos_coeffs=sos, steady_iter=0)
for col in RAW_IMU_COLS:
    df_compare[col + "_h_noinit"] = df_noinit[col + "_h"].to_numpy(np.float64)

print("✅ 双向滤波列 *_bi & 未初始化列 *_h_noinit 已生成")

def _axis_from_col(col):
    # Accel_x/gyro_x -> x
    return str(col).split('_')[-1].lower()

def _pick_fwd_col(df, col):
    # 兼容可能存在的 *_h_py（你的前面绘图有这个逻辑）
    if (col + "_h_py") in df.columns:
        return col + "_h_py"
    return col + "_h"

def plot_phase_compare(group_cols, title, ylabel, ylim=None):
    t = df_compare["time_sec"].to_numpy(np.float64)
    fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
    fig.suptitle(title, fontsize=14, fontweight='bold')
    for i, col in enumerate(group_cols):
        ax = axes[i]
        axis_key = _axis_from_col(col)
        color = axis_colors.get(axis_key, 'blue')
        fwd = _pick_fwd_col(df_compare, col)
        ax.plot(t, df_compare[col], label=f"{col} 原始", color=color, alpha=alpha_raw, lw=0.8)
        ax.plot(t, df_compare[fwd], label=f"{col} 正向滤波(有相位)", color=color, lw=lw_filt, ls='--')
        ax.plot(t, df_compare[col + "_bi"], label=f"{col} 双向滤波(零相位)", color=color, lw=lw_filt, ls='-')
        ax.set_ylabel(col)
        if ylim is not None:
            ax.set_ylim(*ylim)
        ax.grid(True, alpha=0.25)
        ax.legend(ncol=3, fontsize=9)
    axes[-1].set_xlabel("Time (s)")
    plt.tight_layout()
    plt.show()

def plot_init_compare(group_cols, title, ylabel, ylim=None):
    t = df_compare["time_sec"].to_numpy(np.float64)
    fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
    fig.suptitle(title, fontsize=14, fontweight='bold')
    for i, col in enumerate(group_cols):
        ax = axes[i]
        axis_key = _axis_from_col(col)
        color = axis_colors.get(axis_key, 'blue')
        fwd = _pick_fwd_col(df_compare, col)
        noi = col + "_h_noinit"
        ax.plot(t, df_compare[col], label=f"{col} 原始", color=color, alpha=alpha_raw, lw=0.8)
        ax.plot(t, df_compare[fwd], label=f"{col} 正向滤波(初始化{CONFIG['filter']['steady_init_iter']}次)", color=color, lw=lw_filt, ls='-')
        ax.plot(t, df_compare[noi], label=f"{col} 正向滤波(不初始化)", color=color, lw=lw_filt, ls='--')
        ax.set_ylabel(col)
        if ylim is not None:
            ax.set_ylim(*ylim)
        ax.grid(True, alpha=0.25)
        ax.legend(ncol=3, fontsize=9)
    axes[-1].set_xlabel("Time (s)")
    plt.tight_layout()
    plt.show()



# ============================================================
# A) 全时间轴相位对比（保持前面配色逻辑：x蓝 y绿 z红）
# ============================================================
t0 = float(df_compare['time_sec'].iloc[0])
t1 = float(df_compare['time_sec'].iloc[-1])
plot_phase_compare(['Accel_x','Accel_y','Accel_z'], f"加速度计：原始 vs 正向滤波 vs 双向滤波（全时段 {t0:.2f}s~{t1:.2f}s）", ylabel="Accel", ylim=(-1500,1500))
plot_phase_compare(['gyro_x','gyro_y','gyro_z'],  f"陀螺仪：原始 vs 正向滤波 vs 双向滤波（全时段 {t0:.2f}s~{t1:.2f}s）",  ylabel="Gyro",  ylim=(-250,250))

# （可选）相位延迟量化：互相关估计 forward 相对 raw 的延迟，bidirectional 应接近 0
def estimate_delay_sec(x_ref, x_test, fs, max_lag_sec=2.0):
    x0 = np.asarray(x_ref, dtype=np.float64) - np.mean(x_ref)
    x1 = np.asarray(x_test, dtype=np.float64) - np.mean(x_test)
    corr = signal.correlate(x1, x0, mode='full', method='fft')
    lags = signal.correlation_lags(len(x1), len(x0), mode='full')
    max_lag = int(max_lag_sec * fs)
    m = (lags >= -max_lag) & (lags <= max_lag)
    best_lag = lags[m][np.argmax(corr[m])]
    return best_lag / fs

col = 'gyro_x'
fwd = _pick_fwd_col(df_compare, col)
d_fwd = estimate_delay_sec(df_compare[col].to_numpy(np.float64), df_compare[fwd].to_numpy(np.float64), fs)
d_bi  = estimate_delay_sec(df_compare[col].to_numpy(np.float64), df_compare[col+'_bi'].to_numpy(np.float64), fs)
print(f"\n[互相关估计] {col}: 正向滤波相对原始的延迟 ≈ {d_fwd*1000:.1f} ms；双向滤波相对原始的延迟 ≈ {d_bi*1000:.1f} ms")

# ============================================================
# B) 全时间轴初始化影响对比（保持前面配色逻辑）
# ============================================================
plot_init_compare(['Accel_x','Accel_y','Accel_z'], f"加速度计：初始化 vs 不初始化（全时段 {t0:.2f}s~{t1:.2f}s）", ylabel="Accel", ylim=(-1500,1500))
plot_init_compare(['gyro_x','gyro_y','gyro_z'],  f"陀螺仪：初始化 vs 不初始化（全时段 {t0:.2f}s~{t1:.2f}s）",  ylabel="Gyro",  ylim=(-250,250))


In [ ]:
# ============================================================
# 解释与验证：为什么“双向滤波”开头多一个峰？为什么和正向结果不只是相位平移？
# ============================================================
# 结论要点：
# 1) sosfiltfilt = 前向滤波 + 反向滤波（同一个 IIR 用两次）
#    => 幅频响应变成 |H(f)|^2，相当于“滤波器阶数翻倍”，因此波形幅度/形状也会变化，不只是相位差。

# 3) 你的正向实现还做了 steady-state 初始化（把第一个点当 DC 偏置空跑 5000 次），
#    这会显著减小正向滤波的起始瞬态；而 sosfiltfilt 的边界机制与之不同，所以两者开头更不一样。
# ============================================================

# 1) 用频率响应证明：双向不是“相位校正的同一个滤波”，而是 |H|^2
w, h = signal.sosfreqz(sos, worN=4096, fs=fs)
mag_fwd_db = 20*np.log10(np.maximum(np.abs(h), 1e-12))
mag_bi_db  = 20*np.log10(np.maximum(np.abs(h)**2, 1e-12))  # |H|^2

plt.figure(figsize=(14, 4.5))
plt.plot(w, mag_fwd_db, label='正向滤波幅频 |H| (dB)', lw=1.8)
plt.plot(w, mag_bi_db,  label='双向滤波幅频 |H|^2 (dB) = 2*|H|(dB)', lw=1.8)
plt.axvline(CONFIG['filter'].get('lowcut_hz', 0.0), color='gray', ls='--', lw=1)
plt.axvline(CONFIG['filter'].get('highcut_hz', 0.0), color='gray', ls='--', lw=1)
plt.title('幅频响应对比：双向滤波会“更陡/更强”——因此结果不只是相位偏移')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Magnitude (dB)')
plt.grid(True, alpha=0.25)
plt.legend()
plt.xlim(0, 10)  # 你的截止频率都在超低频，放大低频区更直观
plt.ylim(-50,0)
plt.show()

# 2) 用不同 padding 方式复现“开头伪峰/瞬态”差异（只用于解释，不建议用于最终结论）
col = 'gyro_x'
axis_key = _axis_from_col(col)
color = axis_colors.get(axis_key, 'blue')
t = df_compare['time_sec'].to_numpy(np.float64)
x = df_compare[col].to_numpy(np.float64)
fwd_col = _pick_fwd_col(df_compare, col)
y_fwd = df_compare[fwd_col].to_numpy(np.float64)
y_bi_default = df_compare[col + '_bi'].to_numpy(np.float64)

# 不同边界策略
y_bi_even = signal.sosfiltfilt(sos, x, padtype='even')
y_bi_const = signal.sosfiltfilt(sos, x, padtype='constant')
y_bi_pad0 = signal.sosfiltfilt(sos, x, padlen=0)  # 禁用 padding（可能更容易边界失真，只用于对比）

# 聚焦开头 20 秒，看“多出来的峰”通常会集中在边界附近
T_focus = 20.0
m = t <= T_focus

plt.figure(figsize=(14, 5))
plt.plot(t[m], x[m], label='原始', color=color, alpha=alpha_raw, lw=0.8)
plt.plot(t[m], y_fwd[m], label='正向滤波(steady init)', color=color, lw=lw_filt, ls='--')
plt.plot(t[m], y_bi_default[m], label='双向滤波 默认pad(odd)', color=color, lw=lw_filt, ls='-')


#plt.plot(t[m], y_bi_even[m], label='双向滤波 padtype=even', color=color, lw=1.2, ls=':')
#plt.plot(t[m], y_bi_const[m], label='双向滤波 padtype=constant', color=color, lw=1.2, ls='-.')
#plt.plot(t[m], y_bi_pad0[m], label='双向滤波 padlen=0', color=color, lw=1.0, alpha=0.9)

plt.title(f'边界处理对双向滤波开头瞬态的影响（{col}，前 {T_focus:.0f}s）')
plt.xlabel('Time (s)')
plt.grid(True, alpha=0.25)
plt.legend(ncol=2, fontsize=9)
plt.tight_layout()
plt.show()


